preprocessing process:
- we define one schema for all clases (text, label)
- don't join them still
- preprocess each dataset separately (!!!CAREFULT)
- merge into one final dataset
- 

## Preprocessing News

Based on the eda, the news dataset is pretty clean, so the preprocessing should be light and we should make sure that we don't over clean it.

In [9]:
import pandas as pd
from pathlib import Path

In [10]:
NEWS_DIR = Path("../data/raw/news/bbc")

data = []

for category_dir in NEWS_DIR.iterdir():
    if category_dir.is_dir():
        label = category_dir.name
        
        for file in category_dir.glob("*.txt"):
            text = file.read_text(encoding="latin-1")
            data.append({
                "label": label,
                "text": text,
                "filename": file.name
            })

news = pd.DataFrame(data)
news.shape

(2225, 3)

In [11]:
news.head()

,label,text,filename
0,entertainment,Musicians to tackle US red tape\n\nMusicians' ...,289.txt
1,entertainment,"U2's desire to be number one\n\nU2, who have w...",262.txt
2,entertainment,Rocker Doherty in on-stage fight\n\nRock singe...,276.txt
3,entertainment,Snicket tops US box office chart\n\nThe film a...,060.txt
4,entertainment,Ocean's Twelve raids box office\n\nOcean's Twe...,074.txt


In [12]:
# make working copy
news_prep = news.copy()
news_prep.shape

(2225, 3)

In [13]:
# keep only relevant columns which are: docuemnt text, class label and maybe filename for reference
news_prep = news_prep[["filename", "label", "text"]].copy()
news_prep.head()

,filename,label,text
0,289.txt,entertainment,Musicians to tackle US red tape\n\nMusicians' ...
1,262.txt,entertainment,"U2's desire to be number one\n\nU2, who have w..."
2,276.txt,entertainment,Rocker Doherty in on-stage fight\n\nRock singe...
3,060.txt,entertainment,Snicket tops US box office chart\n\nThe film a...
4,074.txt,entertainment,Ocean's Twelve raids box office\n\nOcean's Twe...


In [14]:
# standardize text column just in case, even though eda showed no missing values, it ensures that all rows are strings
news_prep["text"] = news_prep["text"].fillna("").astype(str)

In [15]:
# strip leading/trailing spaces
news_prep["text"] = news_prep["text"].str.strip()

In [16]:
# normalize repeated whitespace
news_prep["text"] = news_prep["text"].str.replace(r"\s+", " ", regex=True).str.strip()

In [17]:
# check duplicates before removing just in case
news_prep["text"].duplicated().sum(), round(news_prep["text"].duplicated().mean() * 100, 2)

(np.int64(100), np.float64(4.49))

In [ ]:
# remove duplicate articles
news_prep = news_prep.drop_duplicates(subset="text").copy()
news_prep.shape

(2125, 3)

In [ ]:
# just to check for empty text after cleaning
(news_prep["text"].str.strip() == "").sum()

np.int64(0)

In [20]:
news_prep["num_words"] = news_prep["text"].str.split().str.len()
news_prep["num_words"].describe()

count    2125.000000
mean      384.115294
std       241.478609
min        89.000000
25%       245.000000
50%       331.000000
75%       471.000000
max      4432.000000
Name: num_words, dtype: float64

In [21]:
news_prep.loc[news_prep["num_words"].sort_values(ascending=False).index[:10], ["filename", "label", "num_words", "text"]]

,filename,label,num_words,text
1822,290.txt,politics,4432,Terror powers expose 'tyranny' The Lord Chance...
382,253.txt,entertainment,3482,Scissor Sisters triumph at Brits US band Sciss...
1636,380.txt,politics,3295,Minimum wage increased to Â£5.05 The minimum w...
1964,401.txt,tech,2969,Losing yourself in online gaming Online role p...
1795,293.txt,politics,2393,Kilroy launches 'Veritas' party Ex-BBC chat sh...
236,353.txt,entertainment,2391,Roundabout continues nostalgia trip The new bi...
298,256.txt,entertainment,2347,"Brits debate over 'urban' music Joss Stone, a ..."
1115,371.txt,sport,1662,All Black magic: New Zealand rugby Playing col...
2096,379.txt,tech,1528,Apple laptop is 'greatest gadget' The Apple Po...
1307,491.txt,sport,1383,Nadal puts Spain 2-0 up Result: Nadal 6-7 (6/8...


In [22]:
news_prep["text"] = news_prep["text"].apply(lambda x: " ".join(x.split()[:500]))

In [23]:
news_prep["num_words"] = news_prep["text"].str.split().str.len()
news_prep["num_words"].describe()

count    2125.000000
mean      343.674824
std       117.626148
min        89.000000
25%       245.000000
50%       331.000000
75%       471.000000
max       500.000000
Name: num_words, dtype: float64

In [24]:
news_prep["doc_type"] = "news"
news_final = news_prep[["filename", "text", "doc_type"]].copy()
news_final.head()

,filename,text,doc_type
0,289.txt,Musicians to tackle US red tape Musicians' gro...,news
1,262.txt,"U2's desire to be number one U2, who have won ...",news
2,276.txt,Rocker Doherty in on-stage fight Rock singer P...,news
3,060.txt,Snicket tops US box office chart The film adap...,news
4,074.txt,Ocean's Twelve raids box office Ocean's Twelve...,news


In [25]:
print(news_final.shape)
print(news_final["doc_type"].value_counts())
print((news_final["text"].str.strip() == "").sum())
print(news_final["text"].duplicated().sum())

(2125, 3)
doc_type
news    2125
Name: count, dtype: int64
0
2


## Preprocessing Emails

Based on the eda, there are many messy stuff:
- huge duplicate problem
- very long outliers
- some extremely short emails
- some missing Message
- oisy header-like structure

In [27]:
emails_prep = pd.read_csv("../data/raw/emails/enron_spam_data.csv")
emails_prep.shape

(33716, 5)

In [28]:
emails_prep = emails_prep.drop(columns=["Unnamed: 0"], errors="ignore")
emails_prep.columns.tolist()

['Subject', 'Message', 'Spam/Ham', 'Date']

In [29]:
emails_prep["Subject"] = emails_prep["Subject"].fillna("").astype(str)
emails_prep["Message"] = emails_prep["Message"].fillna("").astype(str)

emails_prep["text"] = (
    emails_prep["Subject"].str.strip() + " " + emails_prep["Message"].str.strip()
).str.strip()

emails_prep[["Subject", "Message", "text"]].head()

,Subject,Message,text
0,christmas tree farm pictures,,christmas tree farm pictures
1,"vastar resources , inc .","gary , production from the high island larger ...","vastar resources , inc . gary , production fro..."
2,calpine daily gas nomination,- calpine daily gas nomination 1 . doc,calpine daily gas nomination - calpine daily g...
3,re : issue,fyi - see note below - already done .\nstella\...,re : issue fyi - see note below - already done...
4,meter 7268 nov allocation,fyi .\n- - - - - - - - - - - - - - - - - - - -...,meter 7268 nov allocation fyi .\n- - - - - - -...


In [30]:
emails_prep = emails_prep[["Subject", "Message", "Spam/Ham", "Date", "text"]].copy()
emails_prep.head()

,Subject,Message,Spam/Ham,Date,text
0,christmas tree farm pictures,,ham,1999-12-10,christmas tree farm pictures
1,"vastar resources , inc .","gary , production from the high island larger ...",ham,1999-12-13,"vastar resources , inc . gary , production fro..."
2,calpine daily gas nomination,- calpine daily gas nomination 1 . doc,ham,1999-12-14,calpine daily gas nomination - calpine daily g...
3,re : issue,fyi - see note below - already done .\nstella\...,ham,1999-12-14,re : issue fyi - see note below - already done...
4,meter 7268 nov allocation,fyi .\n- - - - - - - - - - - - - - - - - - - -...,ham,1999-12-14,meter 7268 nov allocation fyi .\n- - - - - - -...


In [31]:
emails_prep["text"] = emails_prep["text"].str.replace(r"\s+", " ", regex=True).str.strip()

In [32]:
(emails_prep["text"].str.strip() == "").sum()

np.int64(0)

In [33]:
emails_prep = emails_prep[emails_prep["text"].str.strip() != ""].copy()
emails_prep.shape

(33716, 5)

In [34]:
emails_prep["num_words"] = emails_prep["text"].str.split().str.len()
emails_prep["num_words"].describe()

count    33716.000000
mean       364.762309
std        838.077937
min          1.000000
25%        165.000000
50%        190.000000
75%        365.000000
max      45450.000000
Name: num_words, dtype: float64

In [35]:
emails_prep.loc[emails_prep["num_words"].sort_values().index[:20], ["Subject", "Message", "num_words", "text"]]

,Subject,Message,num_words,text
1655,revised,,1,revised
29109,website,,1,website
7569,fyi,,1,fyi
2537,house pictures,,2,house pictures
3273,gymnastics pictures,,2,gymnastics pictures
7136,elena chilkina,hi,3,elena chilkina hi
13257,password - fast,,3,password - fast
13760,please see attached,,3,please see attached
6974,this,hurricane elana\n,3,this hurricane elana
0,christmas tree farm pictures,,4,christmas tree farm pictures


In [36]:
emails_prep = emails_prep[emails_prep["num_words"] >= 5].copy()
emails_prep.shape

(33700, 6)

In [37]:
emails_prep["text"].duplicated().sum(), round(emails_prep["text"].duplicated().mean() * 100, 2)

(np.int64(18125), np.float64(53.78))

In [38]:
emails_prep = emails_prep.drop_duplicates(subset="text").copy()
emails_prep.shape

(15575, 6)

In [39]:
emails_prep["num_words"] = emails_prep["text"].str.split().str.len()
emails_prep["num_words"].describe()

count    15575.000000
mean       347.581894
std       1085.865992
min          5.000000
25%         72.000000
50%        173.000000
75%        367.000000
max      45450.000000
Name: num_words, dtype: float64

In [40]:
emails_prep.loc[emails_prep["num_words"].sort_values(ascending=False).index[:10], ["Subject", "num_words", "text"]]

,Subject,num_words,text
14254,enron mentions,45450,enron mentions enron : a wake - up call the wa...
13901,enron mentions - 11 / 09 / 01 - 11 / 10 / 01,34912,enron mentions - 11 / 09 / 01 - 11 / 10 / 01 r...
14190,enron mentions,33039,enron mentions fall of a power giant : bailout...
14123,enron mentions - 11 / 24 / 01 - 11 / 25 / 01,28015,enron mentions - 11 / 24 / 01 - 11 / 25 / 01 a...
14146,enron mentions,27037,enron mentions enron and dynegy discuss plan t...
14167,enron mentions,26324,enron mentions don ' t bet it all on your empl...
13588,enron mentions,23343,enron mentions enron discusses credit line of ...
13867,enron mentions,22467,enron mentions enron slashes profits since 199...
13948,enron mentions,21372,enron mentions enron ' s lay turns down severa...
13845,enron mentions,20004,enron mentions dynegy is in talks on purchasin...


In [41]:
emails_prep = emails_prep[emails_prep["num_words"] <= 5000].copy()
emails_prep.shape

(15497, 6)

In [42]:
emails_prep["text"] = emails_prep["text"].apply(lambda x: " ".join(x.split()[:500]))

In [43]:
emails_prep["num_words"] = emails_prep["text"].str.split().str.len()
emails_prep["num_words"].describe()

count    15497.000000
mean       222.273279
std        168.166924
min          5.000000
25%         72.000000
50%        171.000000
75%        363.000000
max        500.000000
Name: num_words, dtype: float64

In [44]:
emails_prep["doc_type"] = "email"

In [45]:
emails_final = emails_prep[["text", "doc_type"]].copy()
emails_final.head()

,text,doc_type
1,"vastar resources , inc . gary , production fro...",email
2,calpine daily gas nomination - calpine daily g...,email
3,re : issue fyi - see note below - already done...,email
4,meter 7268 nov allocation fyi . - - - - - - - ...,email
5,"mcmullen gas for 11 / 99 jackie , since the in...",email


In [46]:
print(emails_final.shape)
print(emails_final["doc_type"].value_counts())
print((emails_final["text"].str.strip() == "").sum())
print(emails_final["text"].duplicated().sum())

(15497, 2)
doc_type
email    15497
Name: count, dtype: int64
0
2


## Preprocessing contracts

In [48]:
CONTRACTS_DIR = Path("../data/raw/contracts/CUAD_v1")
TXT_DIR = CONTRACTS_DIR / "full_contract_txt"
MASTER_CSV = CONTRACTS_DIR / "master_clauses.csv"

data = []

for file in TXT_DIR.glob("*.txt"):
    try:
        text = file.read_text(encoding="utf-8")
    except UnicodeDecodeError:
        text = file.read_text(encoding="latin-1")
    
    data.append({
        "filename": file.name,
        "contract_name": file.stem,
        "text": text
    })

contracts = pd.DataFrame(data)
contracts.shape

(510, 3)

In [49]:
contracts_prep = contracts.copy()
contracts_prep.shape

(510, 3)

In [50]:
contracts_prep = contracts_prep[["filename", "contract_name", "text"]].copy()
contracts_prep.head()

,filename,contract_name,text
0,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,EXHIBIT 10.6\n\n ...
1,"WHITESMOKE,INC_11_08_2011-EX-10.26-PROMOTION A...","WHITESMOKE,INC_11_08_2011-EX-10.26-PROMOTION A...",Exhibit 10.26 CONFIDENTIAL TREATMENT HAS BE...
2,LohaCompanyltd_20191209_F-1_EX-10.16_11917878_...,LohaCompanyltd_20191209_F-1_EX-10.16_11917878_...,Exhibit 10.16 SUPPLY CONTRACT Contract No: Dat...
3,CENTRACKINTERNATIONALINC_10_29_1999-EX-10.3-WE...,CENTRACKINTERNATIONALINC_10_29_1999-EX-10.3-WE...,1 ...
4,NELNETINC_04_08_2020-EX-1-JOINT FILING AGREEME...,NELNETINC_04_08_2020-EX-1-JOINT FILING AGREEMENT,Exhibit 1\n\nJOINT FILING AGREEMENT\n\nThe und...


In [51]:
contracts_prep["text"] = contracts_prep["text"].fillna("").astype(str)

In [52]:
contracts_prep["text"] = contracts_prep["text"].str.strip()

In [53]:
contracts_prep["text"] = (
    contracts_prep["text"]
    .str.replace("&nbsp;", " ", regex=False)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [54]:
(contracts_prep["text"].str.strip() == "").sum()

np.int64(0)

In [55]:
contracts_prep["text"].duplicated().sum(), round(contracts_prep["text"].duplicated().mean() * 100, 2)

(np.int64(1), np.float64(0.2))

In [56]:
contracts_prep = contracts_prep.drop_duplicates(subset="text").copy()
contracts_prep.shape

(509, 3)

In [57]:
contracts_prep["num_words"] = contracts_prep["text"].str.split().str.len()
contracts_prep["num_words"].describe()

count      509.000000
mean      7872.998035
std       8371.268256
min        109.000000
25%       2472.000000
50%       5039.000000
75%      10211.000000
max      47733.000000
Name: num_words, dtype: float64

In [58]:
contracts_prep.loc[
    contracts_prep["num_words"].sort_values(ascending=False).index[:10],
    ["filename", "contract_name", "num_words", "text"]
]

,filename,contract_name,num_words,text
470,"GOOSEHEADINSURANCE,INC_04_02_2018-EX-10.6-Fran...","GOOSEHEADINSURANCE,INC_04_02_2018-EX-10.6-Fran...",47733,"Exhibit 10.6 Goosehead Insurance Agency, LLC F..."
178,PhasebioPharmaceuticalsInc_20200330_10-K_EX-10...,PhasebioPharmaceuticalsInc_20200330_10-K_EX-10...,45650,Exhibit 10.21 Certain information has been exc...
188,"CERES,INC_01_25_2012-EX-10.20-Collaboration Ag...","CERES,INC_01_25_2012-EX-10.20-Collaboration Ag...",45577,Exhibit 10.20 Pages where confidential treatme...
53,VerizonAbsLlc_20200123_8-K_EX-10.4_11952335_EX...,VerizonAbsLlc_20200123_8-K_EX-10.4_11952335_EX...,45143,Exhibit 10.4 FORM OF TRANSFER AND SERVICING AG...
255,"Array BioPharma Inc. - LICENSE, DEVELOPMENT AN...","Array BioPharma Inc. - LICENSE, DEVELOPMENT AN...",42742,[ * ] = Certain confidential information conta...
443,MANUFACTURERSSERVICESLTD_06_05_2000-EX-10.14-O...,MANUFACTURERSSERVICESLTD_06_05_2000-EX-10.14-O...,41962,Exhibit 10.14 OUTSOURCING AGREEMENT BETWEEN IN...
259,RevolutionMedicinesInc_20200117_S-1_EX-10.1_11...,RevolutionMedicinesInc_20200117_S-1_EX-10.1_11...,40936,Exhibit 10.1 [***] Certain information in this...
128,HarpoonTherapeuticsInc_20200312_10-K_EX-10.18_...,HarpoonTherapeuticsInc_20200312_10-K_EX-10.18_...,38518,Exhibit 10.18 Confidential EXECUTION COPY CERT...
142,GRANTIERRAENERGYINC_05_07_2012-EX-10.6-TRANSPO...,GRANTIERRAENERGYINC_05_07_2012-EX-10.6-TRANSPO...,35339,EXHIBIT 10.6 TRANSPORTATION CONTRACT SPECIFIC ...
139,AtnInternationalInc_20191108_10-Q_EX-10.1_1187...,AtnInternationalInc_20191108_10-Q_EX-10.1_1187...,34973,Exhibit 10.1 CERTAIN CONFIDENTIAL PORTIONS OF ...


In [59]:
contracts_prep = contracts_prep[contracts_prep["num_words"] <= 30000].copy()
contracts_prep.shape

(494, 4)

In [60]:
contracts_prep["text"] = contracts_prep["text"].apply(lambda x: " ".join(x.split()[:1500]))

In [61]:
contracts_prep["num_words"] = contracts_prep["text"].str.split().str.len()
contracts_prep["num_words"].describe()

count     494.000000
mean     1385.603239
std       315.071260
min       109.000000
25%      1500.000000
50%      1500.000000
75%      1500.000000
max      1500.000000
Name: num_words, dtype: float64

In [62]:
contracts_prep["doc_type"] = "contract"

In [63]:
contracts_final = contracts_prep[["text", "doc_type"]].copy()
contracts_final.head()

,text,doc_type
0,EXHIBIT 10.6 DISTRIBUTOR AGREEMENT THIS DISTRI...,contract
1,Exhibit 10.26 CONFIDENTIAL TREATMENT HAS BEEN ...,contract
2,Exhibit 10.16 SUPPLY CONTRACT Contract No: Dat...,contract
3,1 Exhibit 10.3 I-on. (LOGO) www.i-on.com 561.3...,contract
4,Exhibit 1 JOINT FILING AGREEMENT The undersign...,contract


In [64]:
print(contracts_final.shape)
print(contracts_final["doc_type"].value_counts())
print((contracts_final["text"].str.strip() == "").sum())
print(contracts_final["text"].duplicated().sum())

(494, 2)
doc_type
contract    494
Name: count, dtype: int64
0
0


## Preprocessing invoices

In [65]:
INVOICE_DIR = Path("../data/raw/invoices/SROIE2019")
TRAIN_DIR = INVOICE_DIR / "train"
TEST_DIR = INVOICE_DIR / "test"

for p in [TRAIN_DIR, TEST_DIR]:
    print(p, p.exists())

../data/raw/invoices/SROIE2019/train True
../data/raw/invoices/SROIE2019/test True


In [67]:
def read_text_file(path):
    return path.read_text(encoding="utf-8", errors="ignore")

def extract_ocr_text_from_box(raw_text):
    lines = str(raw_text).splitlines()
    texts = []
    for line in lines:
        parts = line.split(",", 8)
        if len(parts) == 9:
            texts.append(parts[8].strip())
    return "\n".join(texts).strip()

def load_split(split_dir, split_name):
    rows = []
    box_dir = split_dir / "box"
    entity_dir = split_dir / "entities"
    img_dir = split_dir / "img"

    for box_file in box_dir.glob("*.txt"):
        stem = box_file.stem
        rows.append({
            "split": split_name,
            "doc_id": stem,
            "text": extract_ocr_text_from_box(read_text_file(box_file))
        })

    return pd.DataFrame(rows)

invoices = pd.concat([
    load_split(TRAIN_DIR, "train"),
    load_split(TEST_DIR, "test")
], ignore_index=True)

In [68]:
invoices_prep = invoices.copy()
invoices_prep.shape

(973, 3)

In [69]:
invoices_prep = invoices_prep[["split", "doc_id", "text"]].copy()
invoices_prep.head()

,split,doc_id,text
0,train,X51006555072,DIGI TELECOMMUNICATIONS SDN BHD\n(201283-M)\nL...
1,train,X51006557117,GARDENIA BAKERIES (KL) SDN BHD (139386 X)\nLOT...
2,train,X51005568884,MR. D.I.Y. SDN BHD\n(CO.REG :704427-T)\nLOT 18...
3,train,X51005711441,"RESTORAN WAN SHENG\n002043319-W\nNO.2, JALAN T..."
4,train,X51005806685,ADVANCO COMPANY\nCOMPANY REG. NO. : 725186-V\n...


In [70]:
invoices_prep["text"] = invoices_prep["text"].fillna("").astype(str)

In [71]:
invoices_prep["text"] = invoices_prep["text"].str.strip()

In [72]:
invoices_prep["text"] = invoices_prep["text"].str.replace(r"[ \t]+", " ", regex=True).str.strip()

In [73]:
(invoices_prep["text"].str.strip() == "").sum()

np.int64(0)

In [74]:
invoices_prep = invoices_prep[invoices_prep["text"].str.strip() != ""].copy()
invoices_prep.shape

(973, 3)

In [75]:
invoices_prep["text"].duplicated().sum(), round(invoices_prep["text"].duplicated().mean() * 100, 2)

(np.int64(2), np.float64(0.21))

In [76]:
invoices_prep = invoices_prep.drop_duplicates(subset="text").copy()
invoices_prep.shape

(971, 3)

In [77]:
invoices_prep["num_words"] = invoices_prep["text"].str.split().str.len()
invoices_prep["num_words"].describe()

count    971.000000
mean     115.214212
std       33.044229
min       27.000000
25%       91.000000
50%      109.000000
75%      132.000000
max      240.000000
Name: num_words, dtype: float64

In [78]:
invoices_prep.loc[
    invoices_prep["num_words"].sort_values().index[:10],
    ["doc_id", "split", "num_words", "text"]
]

,doc_id,split,num_words,text
361,X51005442341,train,27,"RESTAURANT SIN DU\nK3-113,JL IBRAHIM SULTAN\n8..."
559,X51008164510,train,43,OCEAN LC PACKAGING ENTERPRISE\nGST NO: 0009459...
33,X51006008197,train,53,RELAIS TOTAL OULMES\nAUTOROUTE RABAT MEKNES\n1...
838,X51006619509,test,54,GUDANG HASIL RESTAURANT SDN BHD\n(1157059-U)\n...
221,X51006466778,train,54,3180502\nTHE\nROTI\nMAN\nBAKERY\n000965911·P\n...
705,X51005442388,test,56,"CONTENTO (JM0761170-K)\n15, JALAN PERMAS 10/7,..."
910,X51006619779,test,58,"YAM FRESH\nNO.145G, JALAN RIMBUNAN RAYA 1,\nLA..."
380,X51005442397,train,58,TRIPLE SIX POINT ENTERPRISE 666\nNO 14& 16 JAL...
836,X51005288570,test,58,3180301\nSECURE PARKING\nCORPORATION S/B\nRIVE...
410,X51005719886,train,59,SUNFISH\n<484297-M>\n22 LRG PERUSAHAAN 4\nKIMP...


In [79]:
invoices_prep["doc_type"] = "invoice"

In [80]:
invoices_final = invoices_prep[["text", "doc_type"]].copy()
invoices_final.head()

,text,doc_type
0,DIGI TELECOMMUNICATIONS SDN BHD\n(201283-M)\nL...,invoice
1,GARDENIA BAKERIES (KL) SDN BHD (139386 X)\nLOT...,invoice
2,MR. D.I.Y. SDN BHD\n(CO.REG :704427-T)\nLOT 18...,invoice
3,"RESTORAN WAN SHENG\n002043319-W\nNO.2, JALAN T...",invoice
4,ADVANCO COMPANY\nCOMPANY REG. NO. : 725186-V\n...,invoice


In [81]:
print(invoices_final.shape)
print(invoices_final["doc_type"].value_counts())
print((invoices_final["text"].str.strip() == "").sum())
print(invoices_final["text"].duplicated().sum())

(971, 2)
doc_type
invoice    971
Name: count, dtype: int64
0
0


In [82]:
print("news_final:", news_final.shape)
print("emails_final:", emails_final.shape)
print("contracts_final:", contracts_final.shape)
print("invoices_final:", invoices_final.shape)

news_final: (2125, 3)
emails_final: (15497, 2)
contracts_final: (494, 2)
invoices_final: (971, 2)


## final check

In [83]:
for name, df in {
    "news": news_final,
    "emails": emails_final,
    "contracts": contracts_final,
    "invoices": invoices_final,
}.items():
    print(f"\n{name}")
    print(df.columns.tolist())
    print("empty texts:", (df["text"].str.strip() == "").sum())
    print("duplicate texts:", df["text"].duplicated().sum())
    print("labels:", df["doc_type"].value_counts().to_dict())


news
['filename', 'text', 'doc_type']
empty texts: 0
duplicate texts: 2
labels: {'news': 2125}

emails
['text', 'doc_type']
empty texts: 0
duplicate texts: 2
labels: {'email': 15497}

contracts
['text', 'doc_type']
empty texts: 0
duplicate texts: 0
labels: {'contract': 494}

invoices
['text', 'doc_type']
empty texts: 0
duplicate texts: 0
labels: {'invoice': 971}


In [84]:
# fixes from the checks above
news_final = news_final.drop_duplicates(subset="text").copy()
news_final = news_final[["text", "doc_type"]].copy()
news_final.shape

emails_final = emails_final.drop_duplicates(subset="text").copy()
emails_final.shape

(15495, 2)

In [85]:
for name, df in {
    "news": news_final,
    "emails": emails_final,
    "contracts": contracts_final,
    "invoices": invoices_final,
}.items():
    print(f"\n{name}")
    print(df.columns.tolist())
    print("empty texts:", (df["text"].str.strip() == "").sum())
    print("duplicate texts:", df["text"].duplicated().sum())


news
['text', 'doc_type']
empty texts: 0
duplicate texts: 0

emails
['text', 'doc_type']
empty texts: 0
duplicate texts: 0

contracts
['text', 'doc_type']
empty texts: 0
duplicate texts: 0

invoices
['text', 'doc_type']
empty texts: 0
duplicate texts: 0


## Merging datasets

In [ ]:
full_df = pd.concat(
    [news_final, emails_final, contracts_final, invoices_final],
    ignore_index=True
)

full_df.shape

In [ ]:
full_df["doc_type"].value_counts()

In [ ]:
(full_df["text"].str.strip() == "").sum()

In [ ]:
full_df["text"].duplicated().sum()